In [1]:
import sqlite3
import pandas as pd

In [2]:
db_path = "../data/raw/statcast.sqlite"

# 2. Query the SQLite master table specifically for VIEWS
query = "SELECT name FROM sqlite_master WHERE type='view' AND name NOT LIKE 'sqlite_%';"

# 3. Read it into pandas
with sqlite3.connect(db_path) as conn:
    views_df = pd.read_sql_query(query, conn)

print("--- Views in Statcast DB ---")
print(views_df)

--- Views in Statcast DB ---
                         name
0     statcast_missed_strikes
1       statcast_missed_balls
2   statcast_missed_balls_100
3               statcast_2016
4               statcast_2017
5               statcast_2018
6               statcast_2019
7               statcast_2020
8               statcast_2021
9               statcast_2022
10              statcast_2023
11              statcast_2024
12              statcast_2025
13              statcast_2026


### Anual Data

In [5]:
years = [
                    #'2016','2017','2018','2019','2020',
                    #'2021','2022','2023',
                    '2024','2025','2026'
                    ]

for year in years:
    query = f"SELECT * FROM statcast_{year}"

    ''' 
    Query details : 
    -- 2016 View
    CREATE VIEW IF NOT EXISTS statcast_2016 AS
    SELECT 
        statcast.*,
        CASE 
            WHEN plate_x BETWEEN -0.7083 AND 0.7083 
            AND plate_z BETWEEN sz_bot AND sz_top 
            AND description = 'ball'
            THEN 1 
            ELSE 0 
        END AS is_missed_strike,
        CASE 
            WHEN (plate_x < -0.7083 OR plate_x > 0.7083
            OR plate_z < sz_bot OR plate_z > sz_top)
            AND description = 'called_strike'
            THEN 1 
            ELSE 0 
        END AS is_missed_ball
    FROM 
        statcast 
    WHERE 
        game_date >= '2016-01-01' 
        AND game_date < '2017-01-01';
    '''

    with sqlite3.connect(db_path) as conn:
        view = pd.read_sql_query(query, conn)

    parquet_path = f"../data/processed/statcast_{year}.parquet"
    view.to_parquet(parquet_path, index=False)

    print(f"Saved {len(view):,} rows to {parquet_path}")

Saved 760,249 rows to ../data/processed/statcast_2024.parquet
Saved 770,795 rows to ../data/processed/statcast_2025.parquet
Saved 653,114 rows to ../data/processed/statcast_2026.parquet


In [ ]:
query = "SELECT * FROM statcast_missed_balls"

''' 
Query details : 
CREATE VIEW IF NOT EXISTS statcast_missed_balls AS
SELECT 
    statcast.*,
    -- Horizontal check: standard MLB home plate width is 17 inches (~0.7083 feet on each side)
    CASE 
        WHEN plate_x NOT BETWEEN -0.7083 AND 0.7083
          OR plate_z NOT BETWEEN sz_bot AND sz_top
        THEN 1 
        ELSE 0 
    END AS is_out_strike_zone
FROM 
    statcast
WHERE 
    description = 'called_strike';
'''

with sqlite3.connect(db_path) as conn:
    view = pd.read_sql_query(query, conn)

parquet_path = "../data/processed/statcast_missed_balls.parquet"
view.to_parquet(parquet_path, index=False)

print(f"Saved {len(view):,} rows to {parquet_path}")



In [ ]:
query = "SELECT * FROM statcast_missed_strikes"

''' 
Query details : 
CREATE VIEW IF NOT EXISTS statcast_missed_strikes AS
SELECT 
    statcast.*,
    -- Horizontal check: standard MLB home plate width is 17 inches (~0.7083 feet on each side)
    CASE 
        WHEN plate_x BETWEEN -0.7083 AND 0.7083 
         AND plate_z BETWEEN sz_bot AND sz_top 
        THEN 1 
        ELSE 0 
    END AS is_in_strike_zone
FROM 
    statcast
WHERE 
    description = 'ball';
'''

with sqlite3.connect(db_path) as conn:
    view = pd.read_sql_query(query, conn)

parquet_path = "../data/processed/statcast_missed_strikes.parquet"
view.to_parquet(parquet_path, index=False)

print(f"Saved {len(view):,} rows to {parquet_path}")